# Encrypted Machine Learning: Training Logic (`ml/training.py`)

This tutorial covers `src/concrete_fhe_toolkit/ml/training.py`. It contains the raw FHE mathematical logic used by trainers. 

For example, `naive_bayes_training` uses a matrix trick to count how many times a feature appeared for a given class entirely under encryption.

## 1. Naive Bayes Training Circuit

Let's see the aggregate statistics circuit for Naive Bayes in action.

In [ ]:
from concrete import fhe
from concrete_fhe_toolkit.ml.training import naive_bayes_training

def test_nb_train(f11: int, f12: int, f21: int, f22: int, l11: int, l12: int, l21: int, l22: int):
    # 2 Samples, 2 Features (binary)
    X_train = [[f11, f12], [f21, f22]]
    # 2 Samples, 2 Classes (one-hot)
    y_train_ohe = [[l11, l12], [l21, l22]]
    return naive_bayes_training(X_train, y_train_ohe)

compiler = fhe.Compiler(test_nb_train, {
    "f11": "encrypted", "f12": "encrypted", "f21": "encrypted", "f22": "encrypted",
    "l11": "encrypted", "l12": "encrypted", "l21": "encrypted", "l22": "encrypted"
})

circuit = compiler.compile([(1, 1, 1, 1, 1, 0, 1, 0)])

# Sample 1: Features [1, 0], Class [1, 0] (Class 0)
# Sample 2: Features [0, 1], Class [0, 1] (Class 1)
feature_counts, class_counts = circuit.encrypt_run_decrypt(
    1, 0, 
    0, 1, 
    1, 0, 
    0, 1
)

# Class counts: 1 sample in Class 0, 1 sample in Class 1
assert list(class_counts) == [1, 1]

# Feature counts:
# Class 0 had Features [1, 0]
# Class 1 had Features [0, 1]
assert list(feature_counts[0]) == [1, 0]
assert list(feature_counts[1]) == [0, 1]

print("✅ Encrypted Naive Bayes Training Logic passed!")